In [66]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [67]:
import numpy as np
import pandas as pd

from pymare import core, datasets, estimators


import phd_project.scripts.data_simulation as dsim
import phd_project.scripts.metaregression_analysis as mra
from phd_project.config import config

cfg = config.load_config()

## 1.0 Datasets

### 1.1 Borenstein Dataset

In [68]:
borenstein_ds1 = {
    "carroll" : {"treated":{"mean": 94, "std": 22, "n": 60},
                 "control":{"mean": 92, "std": 20, "n": 60}},
    "grant" : {"treated":{"mean": 98, "std": 21, "n": 65},
               "control":{"mean": 92, "std": 22, "n": 65}},
    "peck" : {"treated":{"mean": 98, "std": 28, "n": 40},
              "control":{"mean": 88, "std": 26, "n": 40}},
    "donat" : {"treated":{"mean": 94, "std": 19, "n": 200},
               "control":{"mean": 82, "std": 17, "n": 200}},
    "stewart" : {"treated":{"mean": 98, "std": 21, "n": 50},
                 "control":{"mean": 88, "std": 22, "n": 45}},
    "young" : {"treated":{"mean": 96, "std": 21, "n": 85},
               "control":{"mean": 92, "std": 22, "n": 85}},
}

d1_reformed = {k1: {(k2, k3):l3 for k2, l2 in l1.items() for k3, l3 in l2.items()} for k1, l1 in borenstein_ds1.items() 
               }
borenstein_ds1 = pd.DataFrame.from_dict(d1_reformed, orient="index")
borenstein_ds1.set_index(pd.MultiIndex.from_tuples(borenstein_ds1.columns), drop=True)


# calculate the effect size (g)
borenstein_ds1[("computed", "df")] = (borenstein_ds1[("treated", "n")] + (borenstein_ds1[("control", "n")] - 2))
borenstein_ds1[("computed", "S_within")] = np.sqrt((((borenstein_ds1[("treated", "n")] - 1) * borenstein_ds1[("treated", "std")] ** 2 + 
                                       (borenstein_ds1[("control", "n")] - 1) * borenstein_ds1[("control", "std")] ** 2) / 
                                       borenstein_ds1[("computed", "df")]))

borenstein_ds1[("computed", "d")] = (borenstein_ds1[("treated", "mean")] - borenstein_ds1[("control", "mean")]) / borenstein_ds1[("computed", "S_within")]
borenstein_ds1[("computed", "Vd")] = (borenstein_ds1[("treated", "n")] + borenstein_ds1[("control", "n")]) / (borenstein_ds1[("treated", "n")] * borenstein_ds1[("control", "n")]) + borenstein_ds1[("computed", "d")] ** 2 / (2 * (borenstein_ds1[("treated", "n")] + borenstein_ds1[("control", "n")]))
borenstein_ds1[("computed", "J")] = 1 - 3 / (4 * borenstein_ds1[("computed", "df")] - 1)
borenstein_ds1[("computed", "g1")] = borenstein_ds1[("computed", "J")] * borenstein_ds1[("computed", "d")]
borenstein_ds1[("computed", "Vg")] = borenstein_ds1[("computed", "J")] ** 2 * borenstein_ds1[("computed", "Vd")]

bor_df1 = borenstein_ds1["computed"][["g1", "Vg"]]

M_re_target = 0.3582
V_mre_target = 0.0111
SE_mre_target = 0.1052

### 1.2 Fragility Curve Dataset

In [69]:
fc_ds1 = pd.read_csv(cfg["proc_data"]["fragility_curve_dataset"])

## 2.0 Comparison against Borenstein

### 2.1 My Handbuilt Model

In [70]:
# testing my random effects summary effect for two-levels (within and between studies) against Borenstein
M_re, V_mre, SE_mre, study_vals = mra.compute_re_summary_effect2(bor_df1, effect_column="g1", variance_column="Vg")

assert np.round(M_re, 4) == M_re_target
assert np.round(V_mre, 4) == V_mre_target
assert np.round(SE_mre, 4) == SE_mre_target

print("My Model matches Borenstein et al.")

My Model matches Borenstein et al.


### 2.2 PyMare

In [71]:
# create the pymare dataset
dset1_pm = core.Dataset(y="g1", v="Vg", data=bor_df1)
est = estimators.DerSimonianLaird(small_sample_correction="wald").fit_dataset(dset1_pm)
results1_pm = est.summary()
results1_pm_df = results1_pm.to_df()

# testing to make sure the PyMare result is the same as the Borenstein example
assert np.round(results1_pm_df.loc[0, "estimate"], 4) == M_re_target
assert np.round(results1_pm_df.loc[0, "se"], 4) == SE_mre_target 

print(results1_pm_df)
print("PyMare matches Borenstein et al.")

        name  estimate        se   z-score   p-value  -log10(p)  ci_0.025  \
0  intercept  0.358229  0.105244  3.403813  0.000665   3.177491  0.151956   

   ci_0.975  
0  0.564503  
PyMare matches Borenstein et al.


## 3.0 Comparison of My Model against PyMare

Comparison using my A1 analysis data (MSA-SS vs. MSA-FX)

### 3.1 A1 Dataset (MSA-SS - MSA-FX)

In [ ]:
# Number of bootstrap replications. One record sample is drawn per replicate and
# reused at every structure - see 1.1.
K_SAMPLES = 5000

# The IM the fragilities are expressed in (selects the collapse-IML table).
IM_TAG = "AvgSA_03"

BOOTSTRAP_PTH = cfg["proc_data"]["bootstrapping"]

msa_ss_ests = mra.load_estimates(BOOTSTRAP_PTH, IM_TAG, arms=["site_msa"])["site_msa"]
msa_fx_ests = mra.load_estimates(BOOTSTRAP_PTH, IM_TAG, arms=["msa_femap695"])["msa_femap695"]

msa_ss_bootstrap = mra.load_saved_bootstrap(
        BOOTSTRAP_PTH, "site_msa", ("theta", "beta"), None, IM_TAG, "by_site",
        k_samples=K_SAMPLES)

msa_fx_bootstrap = mra.load_saved_bootstrap(
        BOOTSTRAP_PTH, "msa_femap695", ("theta", "beta"), None, IM_TAG, "by_site",
        k_samples=K_SAMPLES)

theta_mss_bootstrap = mra.reformat_bootstrap_df(msa_ss_bootstrap["theta"])
theta_mfx_bootstrap = mra.reformat_bootstrap_df(msa_fx_bootstrap["theta"])

at = np.log(theta_mss_bootstrap)      # theta from site-specific msa
bt = np.log(theta_mfx_bootstrap)      # theta from fixed-set msa
at_hat = np.log(msa_ss_ests["theta"]) 
bt_hat = np.log(msa_fx_ests["theta"])

# Calculate the effect size and variance for each study
# get the difference in log units (same as the ratio in real units)
y1 = at - bt    # mss - mfx     shape(n_rows, k_samples) <-- bootstrap replicates
y1_hat = at_hat - bt_hat      # mss - mfx direct from fragilities (n_rows, 1) <-- estimate of effect size
# calculate the total variance directly from the bootstrap samples which will 
# automatically contain any covariance incase they aren't fully independent. 
v1_boot = y1.var(axis=1)    # variance along a row -> reduce the columns  <-- variance from the bootstrap


df1 = pd.DataFrame.from_dict({"y1":y1_hat, "v1":v1_boot}, orient="columns")
dset1 = core.Dataset(y="y1", v="v1", data=df1)
df1.head()

y1        v1
site n_storeys                    
0    3          0.025260  0.002942
     5          0.064861  0.001253
1    3          0.060150  0.001938
     5          0.122803  0.001206
2    3          0.044743  0.001690

In [73]:
dset1.to_df().head()

,y,v,intercept
0,0.025260,0.002942,1.0
1,0.064861,0.001253,1.0
2,0.060150,0.001938,1.0
3,0.122803,0.001206,1.0
4,0.044743,0.001690,1.0


### 3.2 Model Comparison

In [74]:
# Fit my model
mm_res_a_dl = mra.compute_re_summary_effect2(df1, effect_column="y1", variance_column="v1")

# Fit the PyMare Model
pm_res_a_dl = estimators.DerSimonianLaird(small_sample_correction="wald").fit_dataset(dset1).summary().to_df()

np.testing.assert_allclose(mm_res_a_dl[0], pm_res_a_dl.loc[0, "estimate"], rtol=1e-12)
np.testing.assert_allclose(mm_res_a_dl[2], pm_res_a_dl.loc[0, "se"], rtol=1e-12)

print("My model matches PyMare")

My model matches PyMare


In [75]:
mm_res_a_dl[0]

np.float64(-0.002936192843953571)

### 3.3 Fake-Data Simulation

Because I am building up a complex statistical model, there aren't any easily available python packages that I could use to verify the procedure. As such using fake-data simulation to verify that the modelling procedure can reproduce expected results is a robust way of verifying what is going on and testing my assumptions.

In the data simulation we keep the site specific variances ($v_{i,boot}$ -> $v_i$) because they are a feature of the study. What we will simulate is the size of the effect at each site. This is done by sampling from two normal distributions: $$\alpha_i=m_1+u_i \sim N(0, \tau^2)$$ $$y_i=\alpha_i+\varepsilon_i \sim N(0, v_i)$$ The first equation simulates the difference of the true site-specific effect from the mean true effect (between site variance). The second equation simulates the difference between the site-specific observation and the true site-specific effect (within site variance)

In [77]:
# Fake-data simulation for A1a, via the reusable simulator in data_simulation.py.
#
# The inputs are exactly the inputs of the real fit, with one addition: the TRUE
# values the estimator is supposed to recover. v1_boot is kept, not simulated -- the
# sampling variance is a feature of the study, and A1a is the one rung of the ladder
# where feeding the COMBINED variance down the diagonal is correct (the SS and FX
# arms use disjoint record sets, so v1[i] = v_a[i] + V_bb[g[i], g[i]] exactly).
#
# Everything lives inside the call, so nothing leaks into the notebook namespace.
M_TRUE = 0.98
TAU_TRUE = 0.65
N_SIMS = 500

n = len(v1_boot)
sim_a1a = dsim.simulate_reml_fits(
    beta_true=M_TRUE,
    X=np.ones((n, 1)),                            # intercept only
    V_known=np.diag(v1_boot),                     # diagonal for A1a; NOT for A1b/A1c
    components={"tau_u": (np.eye(n), TAU_TRUE)},  # (factor, true SD)
    n_sims=N_SIMS,
    seed=20260914,
    fit_kwargs={"method": "cholesky"},            # ~3.5x faster at n = 120
)

[simulate_reml_fits] 50/500
[simulate_reml_fits] 100/500
[simulate_reml_fits] 150/500
[simulate_reml_fits] 200/500
[simulate_reml_fits] 250/500
[simulate_reml_fits] 300/500
[simulate_reml_fits] 350/500
[simulate_reml_fits] 400/500
[simulate_reml_fits] 450/500
[simulate_reml_fits] 500/500


In [78]:
sim_a1a.draws.head()

,converged,nll,start_spread,beta_0,se_beta_0,tau2_tau_u,at_zero_tau_u
0,True,14.209250,7.105427e-15,0.827982,0.061149,0.445838,False
1,True,1.505629,7.105427e-15,0.911928,0.054954,0.359526,False
2,True,3.020325,1.421085e-14,0.901990,0.055642,0.368648,False
3,True,6.579716,7.105427e-15,0.929241,0.057324,0.391459,False
4,True,8.785612,1.421085e-14,1.013719,0.058421,0.406691,False


In [85]:
sim_a1a.draws

,converged,nll,start_spread,beta_0,se_beta_0,tau2_tau_u,at_zero_tau_u
0,True,14.209250,7.105427e-15,0.827982,0.061149,0.445838,False
1,True,1.505629,7.105427e-15,0.911928,0.054954,0.359526,False
2,True,3.020325,1.421085e-14,0.901990,0.055642,0.368648,False
3,True,6.579716,7.105427e-15,0.929241,0.057324,0.391459,False
4,True,8.785612,1.421085e-14,1.013719,0.058421,0.406691,False
...,...,...,...,...,...,...,...
495,True,23.077975,1.421085e-14,0.967650,0.065895,0.518179,False
496,True,14.724336,7.105427e-15,0.994844,0.061381,0.449236,False
497,True,22.800284,7.105427e-15,1.107726,0.065720,0.515421,False
498,True,15.632117,0.000000e+00,1.109326,0.061846,0.456123,False


There are several checks that we can carry out:
1. Coverage of $m_1$ --> does $m_1$ land in the right %-ile intervals with the correct frequency
2. Do the SE's match --> analytic (i.e. from the weights) vs. actual (from the std of $m_1$)
4. Calculate the bias in $\tau^2$

In [86]:
# All three checks at once. Read them in this order:
#   z / pct_error  -- is m1 unbiased, and does the bias matter?
#   se_ratio       -- does the model-based SE match the actual spread of m1?
#   cov_68/cov_95  -- do the intervals cover at their nominal rate?
# tau2 carries no SE or coverage by design: fit_reml produces no closed-form SE for
# a variance component, so its uncertainty has to come from the outer bootstrap.
_ = dsim.print_simulation_report(sim_a1a)

500 simulations, n = 120, q = 1, seed = 20260914
components  : tau_u
converged   : 100.0%
start spread: max 2.84e-14 (should be ~0; large means a flat or multi-modal surface)

             truth    mean     bias    mcse        z  pct_error  mean_analytic_se  empirical_sd  se_ratio  cov_68  cov_95  frac_at_zero
parameter                                                                                                                              
beta_0     0.98000 0.97570 -0.00430 0.00258 -1.66489   -0.43851           0.05931       0.05772   1.02762 0.68800 0.96000           NaN
tau2_tau_u 0.42250 0.42111 -0.00139 0.00249 -0.55956   -0.33016               NaN       0.05574       NaN     NaN     NaN       0.00000


### 3.4 Implementing REML

Restricted/Residual Maximum Likelihood is required for estimating the variance components of the complex statisitical models. It is an improvement over the Method-of-moments (DerSimonian-Laird from Borenstein et al.) because:
1. REML can handle unbalanced data. MoM becomes less accurate when the sample sizes are unequal across groups
2. MoM underestimates the between-study variance (Borenstein et al.) which artificially narrows CIs. REML in contrast accounts for Dofs lost when estimating fixed effects. This correction prevents the downward bias of MoM
3. REML is a likelihood based approach and the CIs are obtained automatically from teh information matrix of the likelihood function, wehreas MoM  is mathematically complex and requires awkward approximations when the data structure is complex (as it is in this case)
4. REML is less biased for small samples, which is important here as we don't have that many sites/structures 

Viechtbauer (2005) -> conclusions


There isn't currently a Python function that can deal with the crossed nature of my meta-regression structure. Therefore it needs to be programmed. Below is a description of the problem


**Effect size:** $y$ this is just a vector of my values per site/structure combination

**Covariates for regression:** $X$ is a matrix of the values of the covariates (predictor variables) for each site. size (NxQ) where Q is the number of covariates considered.

**Known Variance Components (shape NxN)**: 
$V_{known}$ is the covariance of the estimation error (within site/structure) in the effect sizes. $$V_{known} = diag(v_i^a)+ZV^{bb}Z^T$$ This equation is the generalised version of $Var(a - b) = v_a^2+b_b^2$ because the simple addition of the variances only holds when all the data points are independent. 

$v_i^a$ is the within site variance for site $i$ from the site specific analysis. 

$V^{bb}$ (shape JxJ) is the covariance of the design groups (i.e. same structures shared across multiple sites) $V^{bb}_{jj´}=Cov(b_j, b_{j´})$ where $b_j$ is the log of $\theta$ or $\beta$ from the MSA-FX analyses (i.e. analyses based on the building group). 

$Z$ (NxJ) is a matrix that maps each building $j$ to a site $i$. $v_i^a$ and $V^{bb}$ are calculated from the bootstrap results.

**Data Structure:**
$G_s$ is a set of arrays that represent the structure of the different variance components that are included in the model. Each array in (NxN). The arrays for each variance component that I consider are as follows:
- $\tau_d^2$ is the between design variance -> ones indicate where designs are shared e.g. 4 site/structure combinations and rows 1 and 3 share a design. Z is (NxJ) where is the number of rows and J the number of designs $$Z=\begin{bmatrix}
1 & 0 & 0 \\
0 & 1 & 0 \\
1 & 0 & 0 \\
0 & 0 & 1
\end{bmatrix}$$ $$ZZ^T = \begin{bmatrix}
1 & 0 & 1 & 0 \\
0 & 1 & 0 & 0 \\
1 & 0 & 1 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}$$ 
- $\tau_s^2$ is the between site variance -> ones indicate where sites are shared across site/structure combinations. 4 site/structure combinations and rows 1 and 2 share a site and 3 and 4 share a different site. S is (NxM) (n_rows by m_sites). $$S = \begin{bmatrix}
1 & 0 \\
1 & 0 \\
0 & 1 \\
0 & 1 
\end{bmatrix}$$ $$SS^T = \begin{bmatrix}
1 & 1 & 0 & 0 \\
1 & 1 & 0 & 0 \\
0 & 0 & 1 & 1 \\
0 & 0 & 1 & 1
\end{bmatrix}$$ 
- $\tau_u^2$ is the remaining unexplained variance due to interaction between the site and the design. This is unique to each site so we just supply and NxN identity matrix.

**The Restricted Log-Likelihood Function**

From Viechtbauer (2005):

$$\ln{L(\sigma^2|y)}=-0.5\ln{|V|}-0.5\ln{|X^TV^{-1}X|}-0.5(y-X\tilde{\beta})^TV^{-1}(y-X\tilde{\beta})$$ where V is the variance matrix (all the variance components combined into one matrix that represents the crossed structure of the data) $$V=C_{known} + \sum{G_p\tau_p^2}$$ This middle term $-0.5\ln{|X^TV^{-1}X|}$ is the bias correction which makes REML better than ML for this type of work.

We can either maximise this function or minimise the negative version of it $$\ln{L(\sigma^2|y)}=0.5[ln{|V|} + \ln{|X^TV^{-1}X|} + (y-X\tilde{\beta})^TV^{-1}(y-X\tilde{\beta})]$$ and the values of $\tau$ (which feed into $V$) which minimise the function are the variance components

In [80]:
fc_ds1.head(10)

,site,n_storeys,structure_id,tag,design_group_id,n_sites_in_group,is_representative,msa_fx_theta,msa_fx_log_theta,msa_fx_var_theta,...,msa_ss_var_log_theta,msa_ss_log_theta_bias,msa_ss_log_theta_bias_corrected,msa_ss_beta,msa_ss_log_beta,msa_ss_var_beta,msa_ss_var_log_beta,msa_ss_log_beta_bias,msa_ss_log_beta_bias_corrected,msa_ss_n_obs
0,0,3,0,3s_cbf_dc2_site0,group_3s_00,8,True,0.411665,-0.887545,0.000436,...,0.002355,0.003918,0.003918,0.273254,-1.297354,0.002489,0.032850,-0.053001,0.0,10
1,1,3,1,3s_cbf_dc2_site1,group_3s_01,6,True,0.392731,-0.934629,0.000187,...,0.000439,-0.010349,-0.010349,0.225325,-1.490210,0.000988,0.018478,0.028381,0.0,10
2,2,3,1,3s_cbf_dc2_site2,group_3s_01,6,False,0.392731,-0.934629,0.000187,...,0.000395,-0.008449,-0.008449,0.229964,-1.469833,0.002258,0.056253,-0.080462,0.0,10
3,3,3,2,3s_cbf_dc2_site3,group_3s_02,1,True,0.402997,-0.908826,0.000277,...,0.000962,-0.006196,-0.006196,0.244984,-1.406560,0.000892,0.016188,-0.074617,0.0,10
4,4,3,0,3s_cbf_dc2_site4,group_3s_00,8,False,0.411665,-0.887545,0.000436,...,0.001109,-0.012423,-0.012423,0.351250,-1.046256,0.003453,0.028328,-0.000616,0.0,10
5,5,3,1,3s_cbf_dc2_site5,group_3s_01,6,False,0.392731,-0.934629,0.000187,...,0.000605,-0.008705,-0.008705,0.212856,-1.547138,0.001021,0.022882,-0.053625,0.0,10
6,6,3,1,3s_cbf_dc2_site6,group_3s_01,6,False,0.392731,-0.934629,0.000187,...,0.000455,-0.000755,-0.000755,0.217329,-1.526343,0.000289,0.007365,-0.048451,0.0,10
7,7,3,3,3s_cbf_dc2_site7,group_3s_03,5,True,0.494296,-0.704621,0.000393,...,0.000313,-0.001695,-0.001695,0.144280,-1.936000,0.000372,0.019446,-0.035633,0.0,10
8,8,3,0,3s_cbf_dc2_site8,group_3s_00,8,False,0.411665,-0.887545,0.000436,...,0.000531,-0.006774,-0.006774,0.167792,-1.785029,0.000439,0.018235,-0.105493,0.0,10
9,9,3,4,3s_cbf_dc2_site9,group_3s_04,3,True,0.386114,-0.951621,0.000230,...,0.000676,-0.005572,-0.005572,0.172803,-1.755606,0.000494,0.017770,-0.045950,0.0,10


In [81]:
# Set up the structured matrices
Z = np.zeros((len(fc_ds1), len(fc_ds1["structure_id"].unique())))
for row, col in enumerate(fc_ds1["structure_id"]):
    Z[row, col] = 1

S = np.zeros((len(fc_ds1), len(fc_ds1["site"].unique())))
for row, col in enumerate(fc_ds1["site"]):
    S[row, col] = 1

### 3.5 A1a Comparison with REML against PyMare 

In [83]:
# My REML Calculation for A1a
y1 = df1["y1"].to_numpy()
X = np.ones(y1.shape)       # ones because I am only interested in the intercept at this stage
known_variance = np.diag(v1_boot)           # C_known --> for A1a comparison with hand calcs this is assumed to be diagonal (independent rows) 
variance_structures = [np.eye(len(y1), len(y1))]    # just interested in the one variance component (tau_u)
tau_names = ["tau_u"]
my_res_a_reml = mra.fit_reml(y1, X, known_variance, variance_structures, tau_names)

# PyMare REML Calculation for A1a
pm_res_a_reml = estimators.VarianceBasedLikelihoodEstimator(method="REML", small_sample_correction="wald").fit_dataset(dset1).summary().to_df()

np.testing.assert_allclose(my_res_a_reml["beta"][0], pm_res_a_reml.loc[0, "estimate"], rtol=1e-6)
np.testing.assert_allclose(my_res_a_reml["se_beta"][0], pm_res_a_reml.loc[0, "se"], rtol=1e-6)

print("My model matches PyMare")

My model matches PyMare


In [84]:
my_res_a_reml

{'beta': array([-0.00290455]),
 'se_beta': array([0.00922404]),
 'vcov_beta': array([[8.50829599e-05]]),
 'tau2': array([0.00756676]),
 'tau': array([0.08698712]),
 'names': ['tau_u'],
 'at_zero_boundary': array([False]),
 'converged': True,
 'nll': -210.30205771263758,
 'start_spread': 1.1368683772161603e-13,
 'method': 'explicit',
 'Sigma': array([[0.01050899, 0.        , 0.        , ..., 0.        , 0.        ,
         0.        ],
        [0.        , 0.00881969, 0.        , ..., 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.00950435, ..., 0.        , 0.        ,
         0.        ],
        ...,
        [0.        , 0.        , 0.        , ..., 0.00946978, 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , ..., 0.        , 0.01319459,
         0.        ],
        [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.00902974]], shape=(120, 120))}